In [ ]:
"""
1 - Dataset + Metadata Loading & Overview -> Yapıyı anlamak
2 - Class Balance (Satır + Dosya) -> Model kararı için
3 - Veri Kalitesi -> Temiz mi değil mi
4 - Zaman Ekseninde Görselleştirme
5 - Key Feature Dağılımları -> normal vs anomaly KDE/Histogram
6 - Data Leakage Risk Analizi

"""

In [1]:
import pandas as pd
from pathlib import Path

# Dataset + Metadata Loading & Overview


In [ ]:
BASE = Path("../data/raw")

# Meta data incelicez ilk önce, iç yapısını ve sütunların ne anlama geldiğini anlamak için.
meta_data_df = pd.read_csv(BASE / "all_metadata.csv")

print("Shape:", meta_data_df.shape)
print("Columns:", meta_data_df.columns.tolist())
print("Dtypes:", meta_data_df.dtypes.tolist())
print("Head:")
meta_data_df.head()

Shape: (1920, 13)
Columns: ['Variant_name', 'Run_Batch', 'Variant_path', 'date_created', 'date_loaded', 'FSW_rows', 'Simulation_columns', 'FSW_columns', 'Scenario', 'Month', 'Time_offset', 'Description', 'data_file']
Dtypes: [dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('O'), dtype('O'), dtype('float64'), dtype('O'), dtype('O')]
Head:


,Variant_name,Run_Batch,Variant_path,date_created,date_loaded,FSW_rows,Simulation_columns,FSW_columns,Scenario,Month,Time_offset,Description,data_file
0,1_0.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_0....,2025-08-29T03:54:34.216196,2025-08-29T03:54:45.984822,602,69,134,baseline,January,0.0,Firesat_Baseline January Time Offset:0.0,./data/raw/baseline/fsw_data_1_0.0_0.csv
1,1_36.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_36...,2025-08-29T03:54:46.089648,2025-08-29T03:54:58.827963,602,69,134,baseline,January,36.0,Firesat_Baseline January Time Offset:36.0,./data/raw/baseline/fsw_data_1_36.0_0.csv
2,1_72.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_72...,2025-08-29T03:54:58.928442,2025-08-29T03:55:11.794282,602,69,134,baseline,January,72.0,Firesat_Baseline January Time Offset:72.0,./data/raw/baseline/fsw_data_1_72.0_0.csv
3,1_108.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_10...,2025-08-29T03:55:11.900084,2025-08-29T03:55:23.651383,602,69,134,baseline,January,108.0,Firesat_Baseline January Time Offset:108.0,./data/raw/baseline/fsw_data_1_108.0_0.csv
4,1_144.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_14...,2025-08-29T03:55:23.744846,2025-08-29T03:55:35.496401,602,69,134,baseline,January,144.0,Firesat_Baseline January Time Offset:144.0,./data/raw/baseline/fsw_data_1_144.0_0.csv


In [17]:
print("Tail:")
meta_data_df.tail()

Tail:


,Variant_name,Run_Batch,Variant_path,date_created,date_loaded,FSW_rows,Simulation_columns,FSW_columns,Scenario,Month,Time_offset,Description,data_file
1915,12_324.0_-50,Firesat_Attack_RWC,Firesat_Attack_RWC/FireSat_updated_12_324.0_-5...,2025-08-29T02:49:55.770782,2025-08-29T02:50:07.523443,602,69,134,attack_rwc,December,324.0,Firesat_Attack_RWC December Time Offset:324.0,./data/raw/attack_rwc/fsw_data_12_324.0_-50.csv
1916,12_324.0_-25,Firesat_Attack_RWC,Firesat_Attack_RWC/FireSat_updated_12_324.0_-2...,2025-08-29T02:50:07.622140,2025-08-29T02:50:19.391689,602,69,134,attack_rwc,December,324.0,Firesat_Attack_RWC December Time Offset:324.0,./data/raw/attack_rwc/fsw_data_12_324.0_-25.csv
1917,12_324.0_0,Firesat_Attack_RWC,Firesat_Attack_RWC/FireSat_updated_12_324.0_0....,2025-08-29T02:50:19.488263,2025-08-29T02:50:31.228495,602,69,134,attack_rwc,December,324.0,Firesat_Attack_RWC December Time Offset:324.0,./data/raw/attack_rwc/fsw_data_12_324.0_0.csv
1918,12_324.0_25,Firesat_Attack_RWC,Firesat_Attack_RWC/FireSat_updated_12_324.0_25...,2025-08-29T02:50:31.327610,2025-08-29T02:50:43.073062,602,69,134,attack_rwc,December,324.0,Firesat_Attack_RWC December Time Offset:324.0,./data/raw/attack_rwc/fsw_data_12_324.0_25.csv
1919,12_324.0_50,Firesat_Attack_RWC,Firesat_Attack_RWC/FireSat_updated_12_324.0_50...,2025-08-29T02:50:43.180588,2025-08-29T02:50:54.919491,602,69,134,attack_rwc,December,324.0,Firesat_Attack_RWC December Time Offset:324.0,./data/raw/attack_rwc/fsw_data_12_324.0_50.csv


In [38]:
#Metadata sütunlarının dağılımlarını inceleyelim.
#Metadata'da senaryo dağılımı:

meta_data_df["Scenario"].value_counts()

Scenario
attack_rwa    600
attack_rwb    600
attack_rwc    600
baseline      120
Name: count, dtype: int64

In [ ]:
# FSW Row metadata içerisindeki 1920 csv dosyasının her birinde kaç satır olduğunu yani her bir csv dosyasının kaç example içerdiğini göseriyor. 1hz sıklıkla kaydedilen yani saniyede bir kaydedilen 10 dakikalık veriler oldukları için hepsinde 60 * 10 + 2 example var. FSW row bunu gösteriyor. 
# Metadata FSW Row:
meta_data_df["FSW_rows"].value_counts()

# Görüldüğü üzere 2 csv dosyasında anormallik var, yani 2 csv dosyasında diğerlerinin aksine 602 satır yok. EDA adımında olduğumuz için müdahele yok ama bu anormalliği kafamızda not aldık.

FSW_rows
602    1918
510       1
341       1
Name: count, dtype: int64

In [ ]:
# FSW Column da metadata içerisindeki 1920 csv dosyasının her birinde kaç feature olduğunu gösteriyor, 600 örneğin feature sayısını veriyor yani.
# Metadata FSW Columns:
meta_data_df["FSW_columns"].value_counts()
# Görüldüğü üzere burada da bazı csv dosyalarında column dengesizliği var, bunlara da ileride bakmak için not aldık.

FSW_columns
134    1726
127     192
90        2
Name: count, dtype: int64

In [ ]:
# Metadata Month dağılımı:
meta_data_df["Month"].value_counts()
# Dengeli sorun yok.

Month
January      160
February     160
March        160
April        160
May          160
June         160
July         160
August       160
September    160
October      160
November     160
December     160
Name: count, dtype: int64

In [ ]:
#Metadata Time Offset:
meta_data_df["Time_offset"].value_counts()
#Dengeli sorun yok.

# Bu incelenecek son sütun değeri idi, kalanları işimize yaramayacak şeyler, şimdi data file'lara bakalım.

Time_offset
0.0      192
36.0     192
72.0     192
108.0    192
144.0    192
180.0    192
216.0    192
252.0    192
288.0    192
324.0    192
Name: count, dtype: int64

In [ ]:
# Then we have the data files themselves. We can reach them by using the paths in the metadata "data_file" column.
#örnek baseline csv dosyası içeriği:
sample_baseline_df = pd.read_csv(BASE / "baseline/fsw_data_1_72.0_0.csv")
sample_baseline_df.head()

# bu şekilde metadata üzerinden data file'lara erişebiliyoruz.

,FswTime(sec),CurrentYear(year),CurrentMonth(month),CurrentDay(day),CurrentHour(hour),CurrentMinute(min),CurrentSecond(sec),Latitude(rad),Longitude(rad),Altitude(m),...,Run_Batch,date_loaded,date_created,UTC,Longitude(deg),Latitude(deg),TimeDelay,Scenario,Run_Batch_Variant,result_label
0,59,2025,1,0,12,19,14,1.27909,-2.33570,900032,...,Firesat_Baseline,2025-08-29 03:55:11.682,2025-08-29 03:55:11.548,2069-12-25 12:18:56,-133.825752,73.286459,72.0,baseline,Firesat_Baseline_1_72.0_0,baseline
1,60,2025,1,0,12,19,15,1.27998,-2.33750,900029,...,Firesat_Baseline,2025-08-29 03:55:11.682,2025-08-29 03:55:11.548,2069-12-25 12:18:57,-133.928885,73.337452,72.0,baseline,Firesat_Baseline_1_72.0_0,baseline
2,61,2025,1,0,12,19,16,1.28087,-2.33930,900029,...,Firesat_Baseline,2025-08-29 03:55:11.682,2025-08-29 03:55:11.548,2069-12-25 12:18:58,-134.032017,73.388445,72.0,baseline,Firesat_Baseline_1_72.0_0,baseline
3,62,2025,1,0,12,19,17,1.28175,-2.34112,900028,...,Firesat_Baseline,2025-08-29 03:55:11.682,2025-08-29 03:55:11.548,2069-12-25 12:18:59,-134.136295,73.438865,72.0,baseline,Firesat_Baseline_1_72.0_0,baseline
4,63,2025,1,0,12,19,18,1.28264,-2.34295,900024,...,Firesat_Baseline,2025-08-29 03:55:11.682,2025-08-29 03:55:11.548,2069-12-25 12:19:00,-134.241147,73.489859,72.0,baseline,Firesat_Baseline_1_72.0_0,baseline


In [ ]:
#örnek attack csv dosyası içeriği:
sample_attack_df = pd.read_csv(BASE / "attack_rwc/fsw_data_12_324.0_25.csv")
sample_attack_df.head()

,FswTime(sec),CurrentYear(year),CurrentMonth(month),CurrentDay(day),CurrentHour(hour),CurrentMinute(min),CurrentSecond(sec),Latitude(rad),Longitude(rad),Altitude(m),...,date_loaded,date_created,UTC,Longitude(deg),Latitude(deg),TimeDelay,AttackIntensity,Scenario,Run_Batch_Variant,result_label
0,59,2025,12,1,13,22,14,-0.564544,-1.49446,900007,...,2025-08-29 02:50:42.962,2025-08-29 02:50:42.838,2071-10-26 13:21:56,-85.626251,-32.345989,324.0,25.0,attack_rwc,Firesat_Attack_RWC_12_324.0_25,baseline
1,60,2025,12,1,13,22,15,-0.563542,-1.49474,900009,...,2025-08-29 02:50:42.962,2025-08-29 02:50:42.838,2071-10-26 13:21:57,-85.642293,-32.288578,324.0,25.0,attack_rwc,Firesat_Attack_RWC_12_324.0_25,baseline
2,61,2025,12,1,13,22,16,-0.562539,-1.49501,900007,...,2025-08-29 02:50:42.962,2025-08-29 02:50:42.838,2071-10-26 13:21:58,-85.657763,-32.231111,324.0,25.0,attack_rwc,Firesat_Attack_RWC_12_324.0_25,baseline
3,62,2025,12,1,13,22,17,-0.561536,-1.49528,900005,...,2025-08-29 02:50:42.962,2025-08-29 02:50:42.838,2071-10-26 13:21:59,-85.673233,-32.173643,324.0,25.0,attack_rwc,Firesat_Attack_RWC_12_324.0_25,baseline
4,63,2025,12,1,13,22,18,-0.560533,-1.49555,900000,...,2025-08-29 02:50:42.962,2025-08-29 02:50:42.838,2071-10-26 13:22:00,-85.688703,-32.116175,324.0,25.0,attack_rwc,Firesat_Attack_RWC_12_324.0_25,baseline


In [46]:
# Add an AttackIntensity column to the metadata dataframe
# Bu şekilde metadata dataframe'ine ekstra sütunlar ekleyebiliyoruz, filtreleme ve analiz yaparken işimize yarayabilecek sütunları yani.

meta_data_df["AttackIntensity"] = meta_data_df["Variant_name"].apply(lambda x: int(x.split("_")[-1]))
meta_data_df.head()

,Variant_name,Run_Batch,Variant_path,date_created,date_loaded,FSW_rows,Simulation_columns,FSW_columns,Scenario,Month,Time_offset,Description,data_file,AttackIntensity
0,1_0.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_0....,2025-08-29T03:54:34.216196,2025-08-29T03:54:45.984822,602,69,134,baseline,January,0.0,Firesat_Baseline January Time Offset:0.0,./data/raw/baseline/fsw_data_1_0.0_0.csv,0
1,1_36.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_36...,2025-08-29T03:54:46.089648,2025-08-29T03:54:58.827963,602,69,134,baseline,January,36.0,Firesat_Baseline January Time Offset:36.0,./data/raw/baseline/fsw_data_1_36.0_0.csv,0
2,1_72.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_72...,2025-08-29T03:54:58.928442,2025-08-29T03:55:11.794282,602,69,134,baseline,January,72.0,Firesat_Baseline January Time Offset:72.0,./data/raw/baseline/fsw_data_1_72.0_0.csv,0
3,1_108.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_10...,2025-08-29T03:55:11.900084,2025-08-29T03:55:23.651383,602,69,134,baseline,January,108.0,Firesat_Baseline January Time Offset:108.0,./data/raw/baseline/fsw_data_1_108.0_0.csv,0
4,1_144.0_0,Firesat_Baseline,Firesat_Baseline/FireSat_baseline_updated_1_14...,2025-08-29T03:55:23.744846,2025-08-29T03:55:35.496401,602,69,134,baseline,January,144.0,Firesat_Baseline January Time Offset:144.0,./data/raw/baseline/fsw_data_1_144.0_0.csv,0


In [ ]:
# Metadata AttackIntensity dağılımı:
meta_data_df["AttackIntensity"].value_counts()

AttackIntensity
 0     480
-50    360
-25    360
 25    360
 50    360
Name: count, dtype: int64

In [ ]:
# Artık metadata'yı her şeyiyle inceledik, sıra baseline, attack_rwa, attack_rwb, attack_rwc dosyalarında. Onların da içeriklerini istatistiklerini vs görüntüleyeceğiz. long_baseline şimdilik bakmayacağım, anoamly detection için gerekli olabilir ama bu klasik classifciaton için şu an gerekli olduğunu düşünmüyorum.

# Baseline inceleyelim şimdi:

